# Relatório final consolidado

Este notebook organiza os resultados finais do projeto sobre sífilis congênita em Porto Alegre, RS, Brasil. Ele consome as views, consultas SQL e imagens já consolidadas, sem repetir processamento pesado.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text

from src.config import DEFAULT_DATABASE_URL, ROOT, load_project_env
from src.visualization.final_report import generate_all

load_project_env()
DATABASE_URL = os.getenv("DATABASE_URL", DEFAULT_DATABASE_URL)
engine = create_engine(DATABASE_URL)
QUERY_DIR = ROOT / "database" / "queries"
FINAL_IMAGES = ROOT / "outputs" / "images" / "final_report"


## Dados e metodologia

O numerador da incidência vem do SINAN/SIFCBR, que contém notificações de sífilis congênita. O denominador vem do SINASC/DNRS, que contém nascidos vivos. A incidência é calculada por 1.000 nascidos vivos, sem linkage individual entre as bases. CNES, SIM e população municipal são usados como contexto agregado.


In [ ]:
indicadores = pd.read_sql_query(
    text((QUERY_DIR / "02_indicadores_municipio.sql").read_text(encoding="utf-8")),
    engine,
)
display(indicadores)


## Incidência e desigualdade racial


In [ ]:
sintese = pd.read_sql_query(
    text((QUERY_DIR / "13_sintese_desigualdade_racial.sql").read_text(encoding="utf-8")),
    engine,
)
display(sintese)


## Pré-natal, diagnóstico e tratamento


In [ ]:
prenatal = pd.read_sql_query(
    text((QUERY_DIR / "06_prenatal_por_grupo_racial.sql").read_text(encoding="utf-8")),
    engine,
)
diagnostico = pd.read_sql_query(
    text((QUERY_DIR / "07_diagnostico_materno_por_grupo_racial.sql").read_text(encoding="utf-8")),
    engine,
)
tratamento = pd.read_sql_query(
    text((QUERY_DIR / "15_tratamento_materno_por_grupo_racial.sql").read_text(encoding="utf-8")),
    engine,
)
display(prenatal.head(12))
display(diagnostico.head(12))
display(tratamento.head(12))


## Perfil socioeconômico e análise interseccional


In [ ]:
perfil = pd.read_sql_query(
    text((QUERY_DIR / "18_perfil_maes_negras_escolaridade_idade.sql").read_text(encoding="utf-8")),
    engine,
)
interseccional = pd.read_sql_query(
    text((QUERY_DIR / "17_analise_interseccional_desigualdade.sql").read_text(encoding="utf-8")),
    engine,
)
display(perfil)
display(interseccional)


## Qualidade dos dados

Categorias ignoradas e sem informação são mantidas nas consultas para medir qualidade e evitar descarte silencioso de registros.


In [ ]:
qualidade = pd.read_sql_query(
    text("""
    SELECT
        base,
        variavel,
        SUM(ignorados) AS ignorados,
        SUM(total) AS total,
        ROUND(SUM(ignorados)::numeric / NULLIF(SUM(total), 0) * 100, 2) AS percentual_ignorado
    FROM gold.qualidade_registros
    WHERE cod_municipio_residencia = '431490'
    GROUP BY base, variavel
    ORDER BY percentual_ignorado DESC
    """),
    engine,
)
display(qualidade)


## Imagens finais exportadas


In [ ]:
imagens = generate_all(DATABASE_URL, "outputs/images/final_report")
for imagem in imagens:
    print(imagem.relative_to(ROOT))


## Conclusão

A evidência central é a persistência de maior incidência estimada entre mães negras em todos os anos analisados. Pré-natal, diagnóstico, tratamento, escolaridade e qualidade dos registros qualificam essa leitura, mas devem ser interpretados como estratos descritivos agregados. O projeto não faz inferência causal nem pareamento individual entre bases.
